# Modelos COMPAS (réplica del paper)

<!-- Este notebook:
- Carga los CSV generados en el notebook de preprocesamiento
- Entrena los modelos del paper:
  - Logistic Regression
  - SVM
  - XGBoost
- Usa los conjuntos de 2, 7 y 8 features
- Calcula métricas:
  - Accuracy
  - F1
  - False Positive / False Negative por raza
- Guarda todo en un archivo JSON -->


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from xgboost import XGBClassifier


## 1. Cargar datos preprocesados

In [2]:
BASE_DIR = Path("data/processed")

df2 = pd.read_csv(BASE_DIR / "compas_features_2.csv") # 2 caracteristicas
df7 = pd.read_csv(BASE_DIR / "compas_features_7.csv") # 7 caracteristicas
df8 = pd.read_csv(BASE_DIR / "compas_preprocessed.csv") # 8 caracteristicas

TARGET = "two_year_recid"

print(df2.shape, df7.shape, df8.shape)


(5270, 3) (5270, 8) (5270, 9)


## 2. Funciones de evaluación

In [3]:
def compute_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    acc = accuracy_score(y_true, y_pred)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = f1_score(y_true, y_pred)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
    }


## 3. Función de entrenamiento

In [4]:
def run_model(model, df, name):
    X = df.drop(columns=[TARGET])
    y = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    metrics = compute_metrics(y_test, preds)
    return {"model": name, "metrics": metrics}


## 4. Ejecutar experimentos

In [5]:
results = []

# Logistic Regression
lr = LogisticRegression(max_iter=2000)

results.append(run_model(lr, df2, "LR_2_features"))
results.append(run_model(lr, df7, "LR_7_features"))
results.append(run_model(lr, df8, "LR_8_features"))

# SVM
svm = SVC(kernel="rbf")

results.append(run_model(svm, df2, "SVM_2_features"))
results.append(run_model(svm, df7, "SVM_7_features"))

# XGBoost
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1)

results.append(run_model(xgb, df8, "XGB_8_features"))

results


[{'model': 'LR_2_features',
  'metrics': {'accuracy': 0.6527514231499051,
   'precision': np.float64(0.6658097686375322),
   'recall': np.float64(0.5232323232323233),
   'f1_score': 0.5859728506787331,
   'true_negatives': np.int64(429),
   'false_positives': np.int64(130),
   'false_negatives': np.int64(236),
   'true_positives': np.int64(259)}},
 {'model': 'LR_7_features',
  'metrics': {'accuracy': 0.6537001897533207,
   'precision': np.float64(0.6518691588785047),
   'recall': np.float64(0.5636363636363636),
   'f1_score': 0.6045503791982665,
   'true_negatives': np.int64(410),
   'false_positives': np.int64(149),
   'false_negatives': np.int64(216),
   'true_positives': np.int64(279)}},
 {'model': 'LR_8_features',
  'metrics': {'accuracy': 0.6574952561669829,
   'precision': np.float64(0.6565420560747663),
   'recall': np.float64(0.5676767676767677),
   'f1_score': 0.6088840736728061,
   'true_negatives': np.int64(412),
   'false_positives': np.int64(147),
   'false_negatives': np.

## 5. Guardar resultados

In [6]:
OUTPUT_FILE = BASE_DIR / "results.json"

with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, indent=2)

print("Resultados guardados en:", OUTPUT_FILE)


TypeError: Object of type int64 is not JSON serializable